# Setup

Import library dan dataset dipakai bersama oleh Task 1 dan Task 2.


In [1]:
# download my recently edited files from GitHub if running this notebook in Google Colab
!mkdir -p src
!wget -N -P src https://raw.githubusercontent.com/teranixbq/SequenceModellingNLP/refs/heads/main/src/char_level.py
!wget -N -P src https://raw.githubusercontent.com/teranixbq/SequenceModellingNLP/refs/heads/main/src/word_level.py

--2026-05-15 12:06:10--  https://raw.githubusercontent.com/teranixbq/SequenceModellingNLP/refs/heads/main/src/char_level.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3882 (3.8K) [text/plain]
Saving to: ‘src/char_level.py’

char_level.py       100%[===================>]   3.79K  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-05-15 12:06:10 (82.4 MB/s) - ‘src/char_level.py’ saved [3882/3882]

--2026-05-15 12:06:10--  https://raw.githubusercontent.com/teranixbq/SequenceModellingNLP/refs/heads/main/src/word_level.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133

In [2]:
import re
from collections import Counter
from pathlib import Path

import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, TensorDataset, random_split

from src.word_level import TextCNN, HierarchicalTextCNN
from src.char_level import CharCNN, HierarchicalCharCNN


## Download dan Baca Dataset


In [3]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

csv_path = next(Path(path).glob("*.csv"))
df = pd.read_csv(csv_path)

reviews = df["review"].tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()


print("Jumlah data:", len(reviews))
print("Contoh label:", labels[0])


Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews
Jumlah data: 50000
Contoh label: 1


# Task 1 - Word-level 1D CNN

## Tokenisasi, Vocabulary, dan Encoding

Review harus diubah menjadi angka: tokenisasi memecah kalimat jadi kata, vocabulary memberi ID untuk setiap kata, encoding mengubah review menjadi deretan ID dengan panjang yang sama.


In [4]:
def tokenize(text):
    return re.findall(r"[A-Za-z0-9']+", text.lower())

def encode_review(text, word_to_id, max_length):
    token_ids = [word_to_id.get(token, word_to_id["<unk>"]) for token in tokenize(text)]
    token_ids = token_ids[:max_length]
    token_ids += [word_to_id["<pad>"]] * (max_length - len(token_ids))
    return token_ids

In [5]:
max_vocab_size = 50000
max_length = 200
batch_size = 32
num_epochs = 3

word_counter = Counter()
for review in reviews:
    word_counter.update(tokenize(review))

word_to_id = {"<pad>": 0, "<unk>": 1}

for word, _ in word_counter.most_common(max_vocab_size - 2):
    word_to_id[word] = len(word_to_id)

vocab_size = len(word_to_id)
print("Vocab size:", vocab_size)


Vocab size: 50000


## Buat DataLoader

In [6]:
encoded_reviews = [encode_review(review, word_to_id, max_length) for review in reviews]

batch_x = torch.tensor(encoded_reviews, dtype=torch.long)
batch_y = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(batch_x, batch_y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

print("Shape X:", batch_x.shape)
print("Shape y:", batch_y.shape)
print("Train data:", len(train_dataset))
print("Validation data:", len(val_dataset))


Shape X: torch.Size([50000, 200])
Shape y: torch.Size([50000])
Train data: 40000
Validation data: 10000


## Compare Pooling Performance

Bagian akhir ini langsung membandingkan `max`, `avg`, `none` atau without pooling, dan optional `adaptive`.


In [7]:
from time import perf_counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if "trained_models" not in globals():
    trained_models = {}


def train_and_evaluate(pooling_type, kernel_sizes = [3, 4]):
    model = TextCNN(
        vocab_size=vocab_size,
        embed_dim=300,
        num_filters=100,
        kernel_sizes=kernel_sizes,
        num_classes=2,
        pooling_type=pooling_type,
        sequence_length=max_length,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(f"Start training pooling: {pooling_type}, Kernel sizes: {kernel_sizes}", flush=True)

    if device == "cuda":
        torch.cuda.synchronize()

    start_time = perf_counter()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_x, batch_y) in enumerate(train_dataloader, start=1):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0 or batch_idx == len(train_dataloader):
                print(
                    f"Pooling: {pooling_type}, Epoch {epoch + 1}/{num_epochs}, "
                    f"Batch {batch_idx}/{len(train_dataloader)}, "
                    f"Loss: {total_loss / batch_idx:.4f}",
                    flush=True,
                )

    if device == "cuda":
        torch.cuda.synchronize()

    training_time = perf_counter() - start_time

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_x, batch_y in val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            y_true += batch_y.cpu().tolist()
            y_pred += predictions.cpu().tolist()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    model_key = f"word_cnn_{pooling_type}_{'_'.join(map(str, kernel_sizes))}"
    trained_models[model_key] = model

    return {
        "model_type": "word_cnn",
        "pooling_type": pooling_type,
        "kernel_sizes": kernel_sizes,
        "model_key": model_key,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "training_time": training_time,
    }


Using device: cuda


In [8]:
results = []

for pooling_type in ["max", "avg", "none", "adaptive"]:
    record = train_and_evaluate(pooling_type)
    results.append(record)

    print(
        f"Model: {record['model_type']}, "
        f"Pooling: {record['pooling_type']}, "
        f"Kernel Sizes: {record['kernel_sizes']}, "
        f"Accuracy: {record['accuracy']:.4f}, "
        f"Precision: {record['precision']:.4f}, "
        f"Recall: {record['recall']:.4f}, "
        f"F1: {record['f1_score']:.4f}, "
        f"Training Time: {record['training_time']:.2f}s",
        flush=True,
    )

Start training pooling: max, Kernel sizes: [3, 4]
Pooling: max, Epoch 1/3, Batch 50/1250, Loss: 0.8444
Pooling: max, Epoch 1/3, Batch 100/1250, Loss: 0.8130
Pooling: max, Epoch 1/3, Batch 150/1250, Loss: 0.7641
Pooling: max, Epoch 1/3, Batch 200/1250, Loss: 0.7283
Pooling: max, Epoch 1/3, Batch 250/1250, Loss: 0.7068
Pooling: max, Epoch 1/3, Batch 300/1250, Loss: 0.6858
Pooling: max, Epoch 1/3, Batch 350/1250, Loss: 0.6696
Pooling: max, Epoch 1/3, Batch 400/1250, Loss: 0.6584
Pooling: max, Epoch 1/3, Batch 450/1250, Loss: 0.6494
Pooling: max, Epoch 1/3, Batch 500/1250, Loss: 0.6388
Pooling: max, Epoch 1/3, Batch 550/1250, Loss: 0.6299
Pooling: max, Epoch 1/3, Batch 600/1250, Loss: 0.6213
Pooling: max, Epoch 1/3, Batch 650/1250, Loss: 0.6137
Pooling: max, Epoch 1/3, Batch 700/1250, Loss: 0.6073
Pooling: max, Epoch 1/3, Batch 750/1250, Loss: 0.6013
Pooling: max, Epoch 1/3, Batch 800/1250, Loss: 0.5960
Pooling: max, Epoch 1/3, Batch 850/1250, Loss: 0.5916
Pooling: max, Epoch 1/3, Batch 90

### Compare World - CNN with 4 kernel sizes

Meenggunaakn kernel size lain untuk membandingkan dengan compare sebelumnya apakah ada perbedaan atau tidak

In [9]:
for pooling_type in ["max", "avg", "none", "adaptive"]:
    record = train_and_evaluate(pooling_type, kernel_sizes=[3, 4, 5, 6])
    results.append(record)

    print(
        f"Model: {record['model_type']}, "
        f"Pooling: {record['pooling_type']}, "
        f"Kernel Sizes: {record['kernel_sizes']}, "
        f"Accuracy: {record['accuracy']:.4f}, "
        f"Precision: {record['precision']:.4f}, "
        f"Recall: {record['recall']:.4f}, "
        f"F1: {record['f1_score']:.4f}, "
        f"Training Time: {record['training_time']:.2f}s",
        flush=True,
    )

Start training pooling: max, Kernel sizes: [3, 4, 5, 6]
Pooling: max, Epoch 1/3, Batch 50/1250, Loss: 0.9262
Pooling: max, Epoch 1/3, Batch 100/1250, Loss: 0.8483
Pooling: max, Epoch 1/3, Batch 150/1250, Loss: 0.8033
Pooling: max, Epoch 1/3, Batch 200/1250, Loss: 0.7749
Pooling: max, Epoch 1/3, Batch 250/1250, Loss: 0.7446
Pooling: max, Epoch 1/3, Batch 300/1250, Loss: 0.7298
Pooling: max, Epoch 1/3, Batch 350/1250, Loss: 0.7123
Pooling: max, Epoch 1/3, Batch 400/1250, Loss: 0.6997
Pooling: max, Epoch 1/3, Batch 450/1250, Loss: 0.6833
Pooling: max, Epoch 1/3, Batch 500/1250, Loss: 0.6712
Pooling: max, Epoch 1/3, Batch 550/1250, Loss: 0.6625
Pooling: max, Epoch 1/3, Batch 600/1250, Loss: 0.6523
Pooling: max, Epoch 1/3, Batch 650/1250, Loss: 0.6440
Pooling: max, Epoch 1/3, Batch 700/1250, Loss: 0.6404
Pooling: max, Epoch 1/3, Batch 750/1250, Loss: 0.6339
Pooling: max, Epoch 1/3, Batch 800/1250, Loss: 0.6261
Pooling: max, Epoch 1/3, Batch 850/1250, Loss: 0.6208
Pooling: max, Epoch 1/3, Ba

### Hierarchical Word CNN

Bagian ini memakai beberapa Conv1D word-level yang disusun bertingkat.

### Train and Evaluate Hierarchical Word CNN

Fungsi ini mengevaluasi `HierarchicalTextCNN` dengan metrik yang sama seperti eksperimen pooling utama.


In [10]:
def train_and_evaluate_hierarchical(kernel_sizes=[3, 5, 7]):
    model = HierarchicalTextCNN(
        vocab_size=vocab_size,
        embed_dim=300,
        num_filters=100,
        kernel_sizes=kernel_sizes,
        num_classes=2,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(f"Start training Hierarchical CNN with max pooling and kernel sizes {kernel_sizes}", flush=True)

    if device == "cuda":
        torch.cuda.synchronize()

    start_time = perf_counter()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_x, batch_y) in enumerate(train_dataloader, start=1):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0 or batch_idx == len(train_dataloader):
                print(
                    f"Hierarchical CNN, Epoch {epoch + 1}/{num_epochs}, "
                    f"Batch {batch_idx}/{len(train_dataloader)}, "
                    f"Loss: {total_loss / batch_idx:.4f}",
                    flush=True,
                )

    if device == "cuda":
        torch.cuda.synchronize()

    training_time = perf_counter() - start_time

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_x, batch_y in val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            y_true += batch_y.cpu().tolist()
            y_pred += predictions.cpu().tolist()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    return {
        "model_type": "hierarchical_word_cnn",
        "pooling_type": "max",
        "kernel_sizes": kernel_sizes,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "training_time": training_time,
    }



### Run Hierarchical Word CNN

In [11]:
if "results" not in globals():
    results = []

hierarchical_record = train_and_evaluate_hierarchical()
results = [
    record
    for record in results
    if record.get("model_type") != "hierarchical_word_cnn"
]
results.append(hierarchical_record)

Start training Hierarchical CNN with max pooling and kernel sizes [3, 5, 7]
Hierarchical CNN, Epoch 1/3, Batch 50/1250, Loss: 0.6980
Hierarchical CNN, Epoch 1/3, Batch 100/1250, Loss: 0.6932
Hierarchical CNN, Epoch 1/3, Batch 150/1250, Loss: 0.6912
Hierarchical CNN, Epoch 1/3, Batch 200/1250, Loss: 0.6833
Hierarchical CNN, Epoch 1/3, Batch 250/1250, Loss: 0.6738
Hierarchical CNN, Epoch 1/3, Batch 300/1250, Loss: 0.6609
Hierarchical CNN, Epoch 1/3, Batch 350/1250, Loss: 0.6498
Hierarchical CNN, Epoch 1/3, Batch 400/1250, Loss: 0.6353
Hierarchical CNN, Epoch 1/3, Batch 450/1250, Loss: 0.6192
Hierarchical CNN, Epoch 1/3, Batch 500/1250, Loss: 0.6071
Hierarchical CNN, Epoch 1/3, Batch 550/1250, Loss: 0.5982
Hierarchical CNN, Epoch 1/3, Batch 600/1250, Loss: 0.5884
Hierarchical CNN, Epoch 1/3, Batch 650/1250, Loss: 0.5817
Hierarchical CNN, Epoch 1/3, Batch 700/1250, Loss: 0.5729
Hierarchical CNN, Epoch 1/3, Batch 750/1250, Loss: 0.5648
Hierarchical CNN, Epoch 1/3, Batch 800/1250, Loss: 0.55

### Task 1 Top 5 Ranking


In [25]:
task1_results = sorted(
    results if "results" in globals() else [],
    key=lambda record: record["accuracy"],
    reverse=True,
)
task1_top5_results = task1_results[:5]
df_result_1_frame = pd.DataFrame(task1_top5_results)
df_result_1 = df_result_1_frame.drop(columns=['model_key'])

df_result_1


,model_type,pooling_type,kernel_sizes,accuracy,precision,recall,f1_score,training_time
0,word_cnn,avg,"[3, 4, 5, 6]",0.8790,0.864685,0.900735,0.882342,67.186841
1,word_cnn,avg,"[3, 4]",0.8782,0.870130,0.891205,0.880541,40.780969
2,word_cnn,adaptive,"[3, 4, 5, 6]",0.8661,0.860008,0.876911,0.868377,65.764987
3,word_cnn,max,"[3, 4]",0.8658,0.892834,0.833631,0.862218,39.266199
4,word_cnn,max,"[3, 4, 5, 6]",0.8621,0.837454,0.901132,0.868127,65.108639


## Task 2 - Character CNN

- Convert text into character sequences (no word tokenization)
- Build character embeddings
- Apply 1D CNN over character sequences to create word-level representations
- Compare the performance of Character CNN vs Word CNN on the same dataset,
- especially on sentences containing slang, typos, or rare words.

### Character Configuration

`char_max_length` membatasi panjang review dalam satuan karakter agar semua input punya shape yang sama.


In [13]:
char_max_length = 500

### Character Vocabulary

Task 2 tidak memakai word tokenization. Setiap review dibaca sebagai urutan karakter, lalu setiap karakter diberi ID.


In [14]:
char_to_id = {"<pad>": 0, "<unk>": 1}

for review in reviews:
    for char in review.lower():
        if char not in char_to_id:
            char_to_id[char] = len(char_to_id)

char_vocab_size = len(char_to_id)
print("Char vocab size:", char_vocab_size)


Char vocab size: 164


### Character Encoding

Review dipotong sampai `char_max_length`, lalu dipadding dengan ID `<pad>` jika lebih pendek.


In [15]:
def encode_review_chars(text, char_to_id, max_length):
    char_ids = [
        char_to_id.get(char, char_to_id["<unk>"])
        for char in text.lower()[:max_length]
    ]
    char_ids += [char_to_id["<pad>"]] * (max_length - len(char_ids))
    return char_ids


encoded_char_reviews = [
    encode_review_chars(review, char_to_id, char_max_length)
    for review in reviews
]

char_batch_x = torch.tensor(encoded_char_reviews, dtype=torch.long)
char_batch_y = torch.tensor(labels, dtype=torch.long)

print("Char X:", char_batch_x.shape)
print("Char y:", char_batch_y.shape)


Char X: torch.Size([50000, 500])
Char y: torch.Size([50000])


### Character DataLoader

Character CNN memakai split train/validation yang sama dengan Word CNN agar perbandingannya fair.


In [16]:
char_dataset = TensorDataset(char_batch_x, char_batch_y)
char_train_dataset = Subset(char_dataset, train_dataset.indices)
char_val_dataset = Subset(char_dataset, val_dataset.indices)

char_train_dataloader = DataLoader(
    char_train_dataset,
    batch_size=batch_size,
    shuffle=True,
)
char_val_dataloader = DataLoader(
    char_val_dataset,
    batch_size=batch_size,
)

print("Char train data:", len(char_train_dataset))
print("Char validation data:", len(char_val_dataset))


Char train data: 40000
Char validation data: 10000


### Character CNN Training Setup

Bagian ini menyiapkan device dan metrik evaluasi untuk Character CNN.


In [17]:
from time import perf_counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if "trained_models" not in globals():
    trained_models = {}


Using device: cuda


### Train and Evaluate Character CNN

In [18]:
def train_and_evaluate_char(pooling_type="max"):
    model_type = "char_cnn"
    kernel_sizes = [3, 5]

    model = CharCNN(
        char_vocab_size=char_vocab_size,
        char_embed_dim=64,
        num_filters=100,
        kernel_sizes=kernel_sizes,
        num_classes=2,
        pooling_type=pooling_type,
        sequence_length=char_max_length,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(
        f"Start training {model_type} with {pooling_type} pooling "
        f"and kernel sizes {kernel_sizes}",
        flush=True,
    )

    if device == "cuda":
        torch.cuda.synchronize()

    start_time = perf_counter()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_x, batch_y) in enumerate(char_train_dataloader, start=1):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0 or batch_idx == len(char_train_dataloader):
                print(
                    f"Model: {model_type}, Epoch {epoch + 1}/{num_epochs}, "
                    f"Batch {batch_idx}/{len(char_train_dataloader)}, "
                    f"Loss: {total_loss / batch_idx:.4f}",
                    flush=True,
                )

    if device == "cuda":
        torch.cuda.synchronize()

    training_time = perf_counter() - start_time

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_x, batch_y in char_val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            y_true += batch_y.cpu().tolist()
            y_pred += predictions.cpu().tolist()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    model_key = f"char_cnn_{pooling_type}_{'_'.join(map(str, kernel_sizes))}"
    trained_models[model_key] = model

    return {
        "model_type": model_type,
        "pooling_type": pooling_type,
        "kernel_sizes": kernel_sizes,
        "model_key": model_key,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "training_time": training_time,
    }


### Run Character CNN

Jalankan Character CNN biasa dengan beberapa pooling type, seperti eksperimen pooling pada Task 1.


In [19]:
task2_results = []

char_results = []

for pooling_type in ["max", "avg", "none", "adaptive"]:
    record = train_and_evaluate_char(pooling_type=pooling_type)
    char_results.append(record)
    task2_results.append(record)

    print(
        f"Model: {record['model_type']}, "
        f"Pooling: {record['pooling_type']}, "
        f"Kernel Sizes: {record['kernel_sizes']}, "
        f"Accuracy: {record['accuracy']:.4f}, "
        f"Precision: {record['precision']:.4f}, "
        f"Recall: {record['recall']:.4f}, "
        f"F1: {record['f1_score']:.4f}, "
        f"Training Time: {record['training_time']:.2f}s",
        flush=True,
    )



Start training char_cnn with max pooling and kernel sizes [3, 5]
Model: char_cnn, Epoch 1/3, Batch 50/1250, Loss: 0.9019
Model: char_cnn, Epoch 1/3, Batch 100/1250, Loss: 0.8416
Model: char_cnn, Epoch 1/3, Batch 150/1250, Loss: 0.8069
Model: char_cnn, Epoch 1/3, Batch 200/1250, Loss: 0.7826
Model: char_cnn, Epoch 1/3, Batch 250/1250, Loss: 0.7638
Model: char_cnn, Epoch 1/3, Batch 300/1250, Loss: 0.7480
Model: char_cnn, Epoch 1/3, Batch 350/1250, Loss: 0.7364
Model: char_cnn, Epoch 1/3, Batch 400/1250, Loss: 0.7252
Model: char_cnn, Epoch 1/3, Batch 450/1250, Loss: 0.7155
Model: char_cnn, Epoch 1/3, Batch 500/1250, Loss: 0.7077
Model: char_cnn, Epoch 1/3, Batch 550/1250, Loss: 0.7005
Model: char_cnn, Epoch 1/3, Batch 600/1250, Loss: 0.6947
Model: char_cnn, Epoch 1/3, Batch 650/1250, Loss: 0.6895
Model: char_cnn, Epoch 1/3, Batch 700/1250, Loss: 0.6837
Model: char_cnn, Epoch 1/3, Batch 750/1250, Loss: 0.6778
Model: char_cnn, Epoch 1/3, Batch 800/1250, Loss: 0.6738
Model: char_cnn, Epoch 1

### Hierarchical Character CNN

Bagian ini memakai `HierarchicalCharCNN`, yaitu beberapa Conv1D karakter yang disusun bertingkat. Jalankan cell ini jika ingin mengambil bonus/eksperimen hierarchical untuk Task 2.


In [20]:
def train_and_evaluate_hierarchical_char(kernel_sizes=[3, 5, 7]):
    model = HierarchicalCharCNN(
        char_vocab_size=char_vocab_size,
        char_embed_dim=64,
        num_filters=100,
        kernel_sizes=kernel_sizes,
        num_classes=2,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(
        f"Start training hierarchical_char_cnn with max pooling "
        f"and kernel sizes {kernel_sizes}",
        flush=True,
    )

    if device == "cuda":
        torch.cuda.synchronize()

    start_time = perf_counter()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (batch_x, batch_y) in enumerate(char_train_dataloader, start=1):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0 or batch_idx == len(char_train_dataloader):
                print(
                    f"Model: hierarchical_char_cnn, Epoch {epoch + 1}/{num_epochs}, "
                    f"Batch {batch_idx}/{len(char_train_dataloader)}, "
                    f"Loss: {total_loss / batch_idx:.4f}",
                    flush=True,
                )

    if device == "cuda":
        torch.cuda.synchronize()

    training_time = perf_counter() - start_time

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_x, batch_y in char_val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            y_true += batch_y.cpu().tolist()
            y_pred += predictions.cpu().tolist()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        "model_type": "hierarchical_char_cnn",
        "pooling_type": "max",
        "kernel_sizes": kernel_sizes,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "training_time": training_time,
    }


hierarchical_char_record = train_and_evaluate_hierarchical_char()

if "task2_results" not in globals():
    task2_results = []

task2_results = [
    record
    for record in task2_results
    if record.get("model_type") != "hierarchical_char_cnn"
]
task2_results.append(hierarchical_char_record)




Start training hierarchical_char_cnn with max pooling and kernel sizes [3, 5, 7]
Model: hierarchical_char_cnn, Epoch 1/3, Batch 50/1250, Loss: 0.6983
Model: hierarchical_char_cnn, Epoch 1/3, Batch 100/1250, Loss: 0.6919
Model: hierarchical_char_cnn, Epoch 1/3, Batch 150/1250, Loss: 0.6808
Model: hierarchical_char_cnn, Epoch 1/3, Batch 200/1250, Loss: 0.6699
Model: hierarchical_char_cnn, Epoch 1/3, Batch 250/1250, Loss: 0.6636
Model: hierarchical_char_cnn, Epoch 1/3, Batch 300/1250, Loss: 0.6545
Model: hierarchical_char_cnn, Epoch 1/3, Batch 350/1250, Loss: 0.6509
Model: hierarchical_char_cnn, Epoch 1/3, Batch 400/1250, Loss: 0.6431
Model: hierarchical_char_cnn, Epoch 1/3, Batch 450/1250, Loss: 0.6380
Model: hierarchical_char_cnn, Epoch 1/3, Batch 500/1250, Loss: 0.6317
Model: hierarchical_char_cnn, Epoch 1/3, Batch 550/1250, Loss: 0.6280
Model: hierarchical_char_cnn, Epoch 1/3, Batch 600/1250, Loss: 0.6249
Model: hierarchical_char_cnn, Epoch 1/3, Batch 650/1250, Loss: 0.6199
Model: hie

## Compare Word CNN vs Character CNN

Bagian ini membandingkan model Word CNN dan Character CNN pada contoh kalimat slang, typo, atau rare words. Cell ini dijalankan setelah eksperimen Word CNN dan Character CNN selesai, karena membutuhkan model yang sudah tersimpan di `trained_models`.


In [24]:
word_source = results if "results" in globals() else []
char_source = char_results if "char_results" in globals() else []
model_store = trained_models if "trained_models" in globals() else {}

word_candidates = [
    record
    for record in word_source
    if record.get("model_type") == "word_cnn" and record.get("model_key") in model_store
]
char_candidates = [
    record
    for record in char_source
    if record.get("model_type") == "char_cnn" and record.get("model_key") in model_store
]

if not word_candidates or not char_candidates:
    raise ValueError("Run Word CNN and Character CNN training cells first.")

best_word_record = max(word_candidates, key=lambda record: record["accuracy"])
best_char_record = max(char_candidates, key=lambda record: record["accuracy"])

best_word_model = model_store[best_word_record["model_key"]]
best_char_model = model_store[best_char_record["model_key"]]

slangtypodata = [
    "omg this movie was sickkkk, def 10/10",
    "the ending made me cry fr fr, so emotional",
    "worst cinemax experience ever, rly bad acting",
    "ngl the cgi was kinda sus and cheap",
    "totally mindblown! dat plot twist was cray",
    "idk why ppl hate this, it was p cool",
    "just watched it and wowww, absolute masterpiece",
    "boring af, i almost fell asleep in d theater",
    "the soundtrack was a whole vibe, luvvv it",
    "istg this is the best film of 2026, no cap"
]


def predict_review(model, encoded_review):
    model.eval()
    input_tensor = torch.tensor([encoded_review], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1).squeeze(0)
        prediction = probabilities.argmax().item()

    return prediction, probabilities[prediction].item()


comparison_rows = []

for review in slangtypodata:
    word_encoded = encode_review(review, word_to_id, max_length)
    char_encoded = encode_review_chars(review, char_to_id, char_max_length)

    word_prediction, word_confidence = predict_review(best_word_model, word_encoded)
    char_prediction, char_confidence = predict_review(best_char_model, char_encoded)

    comparison_rows.append({
        "review": review,
        "word_prediction": "positive" if word_prediction == 1 else "negative",
        "word_confidence": word_confidence,
        "char_prediction": "positive" if char_prediction == 1 else "negative",
        "char_confidence": char_confidence,
        "word_model": best_word_record["model_key"],
        "char_model": best_char_record["model_key"],
    })

print("Word CNN vs Character CNN")
slang_typo_rare_comparison = pd.DataFrame(comparison_rows)
slang_typo_rare_comparison


Word CNN vs Character CNN


,review,word_prediction,word_confidence,char_prediction,char_confidence,word_model,char_model
0,"omg this movie was sickkkk, def 10/10",positive,0.634045,positive,0.557419,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
1,"the ending made me cry fr fr, so emotional",positive,0.661720,positive,0.601238,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
2,"worst cinemax experience ever, rly bad acting",negative,0.976381,negative,0.954350,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
3,ngl the cgi was kinda sus and cheap,negative,0.651753,negative,0.686422,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
4,totally mindblown! dat plot twist was cray,negative,0.521632,negative,0.523562,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
5,"idk why ppl hate this, it was p cool",negative,0.550403,positive,0.559360,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
6,"just watched it and wowww, absolute masterpiece",positive,0.775537,positive,0.603846,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
7,"boring af, i almost fell asleep in d theater",negative,0.909897,negative,0.628657,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
8,"the soundtrack was a whole vibe, luvvv it",positive,0.581065,positive,0.692707,word_cnn_avg_3_4_5_6,char_cnn_max_3_5
9,"istg this is the best film of 2026, no cap",positive,0.733415,negative,0.569433,word_cnn_avg_3_4_5_6,char_cnn_max_3_5


### Task 2 Top 5 Ranking


In [26]:
if "task2_results" not in globals() or not task2_results:
    raise ValueError("Run Task 2 training cells first.")

task2_results = sorted(
    task2_results,
    key=lambda record: record["accuracy"],
    reverse=True,
)
task2_top5_results = task2_results[:5]

df_result_2_frame = pd.DataFrame(task2_top5_results)
df_result_2 = df_result_2_frame.drop(columns=['model_key'])

df_result_2


,model_type,pooling_type,kernel_sizes,accuracy,precision,recall,f1_score,training_time
0,char_cnn,max,"[3, 5]",0.7722,0.747889,0.826285,0.785135,17.106889
1,hierarchical_char_cnn,max,"[3, 5, 7]",0.7599,0.708347,0.889617,0.788700,27.866008
2,char_cnn,adaptive,"[3, 5]",0.7418,0.870734,0.572365,0.690704,16.200598
3,char_cnn,avg,"[3, 5]",0.7185,0.704453,0.759976,0.731162,19.208938
4,char_cnn,none,"[3, 5]",0.7082,0.695300,0.748858,0.721086,17.538617
